# PyTorch Experience · 03
## `model.train()` vs `model.eval()`

**One glance:** These methods select the model's **mode**. Neither runs a forward pass or updates weights.

**Official docs:** [Module.train()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train) · [Module.eval()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval)

[← Pytorch Experience](https://lawson-dong.vercel.app/coding/pytorch-experience)

### 1 · Choose a mode, then call the model

![Training versus evaluation mode workflow](https://raw.githubusercontent.com/Lawson-Dong/personal-website/main/public/images/pytorch-train-eval-flow.svg)

| Method | Purpose | Runs forward? |
|---|---|---|
| `model.train()` | Enable training behavior | **No** |
| `model.eval()` | Enable evaluation behavior | **No** |
| `model(x)` | Compute the output using the current mode | **Yes** |

**Remember:** `model.train()` is the same as `model.train(True)`. `model.eval()` is the same as `model.train(False)`. `model.train(x)` and `model.eval(x)` are **not** how you pass input tensors.

### 2 · A tiny Dropout example

In training mode, Dropout randomly removes values. In evaluation mode, Dropout becomes an identity operation. We choose `p=1.0` just to make the output obvious.

In [ ]:
import torch
from torch import nn

model = nn.Dropout(p=1.0)
x = torch.ones(4)

model.train()
print('Training:', model(x))  # tensor([0., 0., 0., 0.])

model.eval()
print('Evaluation:', model(x))  # tensor([1., 1., 1., 1.])

### 3 · The `training` flag

`model.eval()` changes `model.training` to `False`, not the model's weights.

In [ ]:
model.train()
print('After train():', model.training)  # True

model.eval()
print('After eval(): ', model.training)  # False

### 4 · Train vs. evaluate: what actually changes?

| Layer | Training mode | Evaluation mode |
|---|---|---|
| `nn.Linear`, `nn.Conv2d` | Same mathematical operation | Same mathematical operation |
| `nn.Dropout` | Randomly masks values | Passes values through |
| `nn.BatchNorm2d` | Usually uses batch statistics and updates running statistics | Usually uses saved running statistics |

The BatchNorm description assumes the usual default `track_running_stats=True`.

**Docs:** [Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html) · [BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)

### 5 · `eval()` is not `inference_mode()`

```python
model.eval()                     # select evaluation behavior
with torch.inference_mode():    # avoid autograd bookkeeping
    output = model(x)           # perform the forward pass
```

`eval()` does **not** turn off gradient tracking. `inference_mode()` does **not** change the model's mode. Also, **`train()` does not train weights by itself**: an actual training loop needs a loss, backpropagation, and an optimizer step.

**Docs:** [torch.inference_mode()](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html) · [A typical optimization loop](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

### Remember

- **`train()`** = training behavior, not an automatic training loop.
- **`eval()`** = evaluation behavior, not a forward pass.
- **`model(x)`** = actually runs the model.
- **`inference_mode()`** = separate switch for autograd-related overhead.

**Official references:** [Module.train()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train) · [Module.eval()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) · [Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html) · [BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)